In [0]:
%python

import pyspark.sql.functions as F

# ============================================================
# 1. READ SOURCE TABLES
# ============================================================

df_orders = spark.table("ecom_dev.silver.orders").alias("orders")
df_customers = spark.table("ecom_dev.gold.dim_customer").alias("cust")
df_products = spark.table("ecom_dev.gold.dim_product").alias("prod")
df_state_region = spark.table("ecom_dev.gold.map_state_region").alias("mapreg")
df_regions = spark.table("ecom_dev.gold.dim_region").alias("reg")


# ============================================================
# 2. JOIN ORDERS WITH CUSTOMERS
#    Bring customer_id and customer state
# ============================================================

orders_customers = (
    df_orders
    .join(
        df_customers.select(
            F.col("customer_id").alias("cust_id"),
            F.col("state").alias("cust_state")
        ),
        df_orders["customer_id"] == F.col("cust_id"),
        how="left"
    )
)


# ============================================================
# 3. MAP CUSTOMER STATE TO REGION
# ============================================================

orders_customers_region = (
    orders_customers
    .join(
        df_state_region.select(
            F.col("state").alias("region_state"),
            F.col("region_id")
        ),
        F.upper(F.trim(F.col("cust_state"))) ==
        F.upper(F.trim(F.col("region_state"))),
        how="left"
    )
)


# ============================================================
# 4. JOIN PRODUCT AND REGION
# ============================================================

df_fact_orders = (
    orders_customers_region

    # Product join
    .join(
        df_products.select(
            F.col("product_id").alias("prod_product_id")
        ),
        orders_customers_region["product_id"] ==
        F.col("prod_product_id"),
        how="left"
    )

    # Region join
    .join(
        df_regions.select(
            F.col("region_id").alias("reg_region_id")
        ),
        orders_customers_region["region_id"] ==
        F.col("reg_region_id"),
        how="left"
    )

    # Final Fact columns
    .select(
        orders_customers_region["order_id"],

        # Customer ID used as customer key
        orders_customers_region["customer_id"].alias("customer_key"),

        # Product ID used as product key
        orders_customers_region["product_id"].alias("product_key"),

        # Region ID used as region key
        orders_customers_region["region_id"].alias("region_key"),

        orders_customers_region["order_date"],
        orders_customers_region["quantity"],
        orders_customers_region["total_amount"]
    )
)


# ============================================================
# 5. DATA QUALITY FILTERS
# ============================================================

df_fact_orders = (
    df_fact_orders
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("customer_key").isNotNull())
    .filter(F.col("product_key").isNotNull())
    .filter(F.col("region_key").isNotNull())
    .filter(F.col("quantity") > 0)
    .filter(F.col("total_amount") >= 0)
    .dropDuplicates(["order_id"])
)


# ============================================================
# 6. WRITE GOLD FACT TABLE
# ============================================================

(
    df_fact_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecom_dev.gold.fact_orders")
)


# ============================================================
# 7. SUCCESS VALIDATION
# ============================================================

print("====================================================")
print("SUCCESS: GOLD FACT TABLE CREATED")
print("====================================================")

print("Table : ecom_dev.gold.fact_orders")

print("Row Count :", df_fact_orders.count())

print("\nSchema:")
df_fact_orders.printSchema()

print("\nSample Data:")
display(df_fact_orders)